# CSV 파일 로더

CSV(Comma-Separated Values)는 데이터를 교환하기 위한 가장 널리 사용되는 파일 형식 중 하나입니다.

각 행이 하나의 레코드를 나타내며, 쉼표(또는 다른 구분자)로 필드를 구분합니다.


## 학습 목표

- CSVLoader를 사용한 CSV 파일 로딩
- CSV 파싱 옵션 커스터마이징 (구분자, 필드명 등)
- 특정 컬럼을 메타데이터로 활용하는 방법


In [1]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()

# CSV 파일 경로
FILE_PATH = "./data/titanic.csv"

# 데이터 파일 경로 확인
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {FILE_PATH}\n현재 디렉토리: {os.getcwd()}")

## 1. 기본 CSV 로딩

`CSVLoader`는 CSV 파일의 각 행을 하나의 Document 객체로 변환합니다.

### 주요 특징

- **행 단위 처리**: 각 행이 별도의 Document
- **컬럼 자동 인식**: 첫 행을 헤더로 인식
- **메타데이터 포함**: 파일 정보를 자동으로 메타데이터에 추가


> 🔑 **핵심 개념**: `CSVLoader`는 CSV의 각 **행(row)**을 하나의 Document로 변환합니다. 열(column)이 아닌 행 단위라는 점이 처음에 헷갈릴 수 있으니 주의하세요.

In [2]:
from langchain_community.document_loaders import CSVLoader

# ============================================================
# TODO: CSVLoader로 CSV 파일을 로딩해보세요
# 힌트: CSVLoader(file_path=FILE_PATH)로 초기화 후 load()를 호출합니다
# 예상 결과: 총 행 수와 첫 번째 행의 내용이 출력됩니다
# ============================================================

# 1단계: CSVLoader 초기화
# - CSVLoader: CSV의 각 행을 하나의 Document로 변환 (헤더는 자동 인식)
loader = CSVLoader(file_path=FILE_PATH)

# 2단계: CSV 로딩
docs = loader.load()

print("=" * 60)
print("📊 CSV 로딩 결과")
print("=" * 60)
print(f"총 행 수 (Document 개수): {len(docs)}")
print(f"\n첫 번째 행 (Document):")
print(docs[0].page_content)

📊 CSV 로딩 결과
총 행 수 (Document 개수): 891

첫 번째 행 (Document):
PassengerId: 1
Survived: 0
Pclass: 3
Name: Braund, Mr. Owen Harris
Sex: male
Age: 22
SibSp: 1
Parch: 0
Ticket: A/5 21171
Fare: 7.25
Cabin: 
Embarked: S


In [3]:
# 메타데이터 확인
print("=" * 60)
print("📋 메타데이터")
print("=" * 60)
for key, value in docs[0].metadata.items():
    print(f"{key}: {value}")


📋 메타데이터
source: ./data/titanic.csv
row: 0


## 2. CSV 파싱 커스터마이징

CSV 파일의 구분자, 인용 부호, 필드명 등을 커스터마이징할 수 있습니다.

### csv_args 주요 옵션

| 옵션 | 설명 | 기본값 |
|------|------|--------|
| `delimiter` | 필드 구분자 | `,` |
| `quotechar` | 인용 부호 문자 | `"` |
| `fieldnames` | 커스텀 필드명 리스트 | 첫 행 사용 |
| `skip initial_space` | 구분자 뒤 공백 무시 | `False` |


In [4]:
# ============================================================
# TODO: csv_args의 fieldnames 옵션으로 한국어 컬럼명을 지정해보세요
# 힌트: CSVLoader(file_path, csv_args={"fieldnames": [...]})
# 예상 결과: 한국어 컬럼명으로 구성된 Document 내용이 출력됩니다
# ============================================================

# 1단계: 커스텀 필드명으로 CSV 로딩
# - csv_args: Python csv 모듈의 DictReader에 전달되는 옵션
# - fieldnames: 첫 행(헤더)을 대체할 커스텀 컬럼명 리스트
loader = CSVLoader(
    file_path=FILE_PATH,
    csv_args={
        "delimiter": ",",           # 쉼표 구분자
        "quotechar": '"',           # 큰따옴표로 인용
        "fieldnames": [             # 한국어 필드명
            "승객ID",
            "생존여부",
            "객실등급",
            "이름",
            "성별",
            "나이",
            "형제자매수",
            "부모자녀수",
            "티켓번호",
            "요금",
            "객실번호",
            "승선항구"
        ],
    },
)

# 2단계: 데이터 로드
custom_docs = loader.load()

print("=" * 60)
print("📊 커스텀 필드명으로 로딩된 데이터")
print("=" * 60)
print(f"첫 번째 행:")
print(custom_docs[1].page_content)  # [1]번 사용 (0번은 원래 헤더 행)

📊 커스텀 필드명으로 로딩된 데이터
첫 번째 행:
승객ID: 1
생존여부: 0
객실등급: 3
이름: Braund, Mr. Owen Harris
성별: male
나이: 22
형제자매수: 1
부모자녀수: 0
티켓번호: A/5 21171
요금: 7.25
객실번호: 
승선항구: S


> 🎯 **강의 포인트**: `source_column`은 RAG 검색 후 "이 답변은 어떤 레코드에서 왔나?"를 추적할 때 핵심입니다. 고객 데이터, 상품 DB 등에서 ID나 이름 컬럼을 source로 지정하면 출처 추적이 쉬워집니다.

## 3. 특정 컬럼을 메타데이터로 사용

`source_column` 옵션을 사용하면 특정 컬럼을 Document의 소스 메타데이터로 지정할 수 있습니다.


In [5]:
# ============================================================
# TODO: source_column 옵션으로 특정 컬럼을 메타데이터 source로 설정하세요
# 힌트: CSVLoader(file_path=FILE_PATH, source_column="Name")
# 예상 결과: 각 Document의 metadata["source"]가 승객 이름으로 출력됩니다
# ============================================================

# 1단계: source_column 지정
# - source_column: 지정된 컬럼 값이 metadata["source"]로 저장됨 (출처 추적에 유용)
loader = CSVLoader(
    file_path=FILE_PATH,
    source_column="Name"  # Name 컬럼을 source 메타데이터로 사용
)

docs_with_source = loader.load()

print("=" * 60)
print("📋 source_column 적용 결과")
print("=" * 60)
print(f"세 번째 승객:")
print(f"소스: {docs_with_source[2].metadata.get('source')}")
print(f"\n내용:")
print(docs_with_source[2].page_content)

📋 source_column 적용 결과
세 번째 승객:
소스: Heikkinen, Miss. Laina

내용:
PassengerId: 3
Survived: 1
Pclass: 3
Name: Heikkinen, Miss. Laina
Sex: female
Age: 26
SibSp: 0
Parch: 0
Ticket: STON/O2. 3101282
Fare: 7.925
Cabin: 
Embarked: S


## 4. 실전 활용: CSV 데이터 전처리

CSV 데이터를 로딩한 후 추가 처리를 할 수 있습니다.


In [6]:
# CSV 데이터 필터링 및 변환
loader = CSVLoader(file_path=FILE_PATH)
all_docs = loader.load()

# 예: 생존자만 필터링
survivors = [
    doc for doc in all_docs 
    if "Survived: 1" in doc.page_content
]

print("=" * 60)
print("📊 데이터 필터링 예제")
print("=" * 60)
print(f"전체 승객 수: {len(all_docs)}")
print(f"생존자 수: {len(survivors)}")
print(f"생존율: {len(survivors) / len(all_docs) * 100:.1f}%")

# 생존자 예시
print(f"\n생존자 예시:")
print(survivors[0].page_content)


📊 데이터 필터링 예제
전체 승객 수: 891
생존자 수: 342
생존율: 38.4%

생존자 예시:
PassengerId: 2
Survived: 1
Pclass: 1
Name: Cumings, Mrs. John Bradley (Florence Briggs Thayer)
Sex: female
Age: 38
SibSp: 1
Parch: 0
Ticket: PC 17599
Fare: 71.2833
Cabin: C85
Embarked: C


## 5. 다양한 구분자 처리

CSV는 쉼표뿐만 아니라 탭, 세미콜론 등 다양한 구분자를 사용할 수 있습니다.


> ⚠️ **자주 하는 실수**: 한글 CSV를 `CSVLoader`로 로딩할 때 `UnicodeDecodeError`가 자주 발생합니다. `encoding="cp949"` 또는 `encoding="utf-8-sig"` (BOM 포함 UTF-8)를 명시적으로 지정하세요.

In [7]:
# 다양한 구분자 예제
print("💡 다양한 구분자 사용 예:")
print()
print("# 탭(TSV) 파일")
print("loader = CSVLoader(")
print("    file_path='data.tsv',")
print("    csv_args={'delimiter': '\\t'}")
print(")")
print()
print("# 세미콜론 구분자")
print("loader = CSVLoader(")
print("    file_path='data.csv',")
print("    csv_args={'delimiter': ';'}")
print(")")
print()
print("# 파이프(|) 구분자")
print("loader = CSVLoader(")
print("    file_path='data.csv',")
print("    csv_args={'delimiter': '|'}")
print(")")


💡 다양한 구분자 사용 예:

# 탭(TSV) 파일
loader = CSVLoader(
    file_path='data.tsv',
    csv_args={'delimiter': '\t'}
)

# 세미콜론 구분자
loader = CSVLoader(
    file_path='data.csv',
    csv_args={'delimiter': ';'}
)

# 파이프(|) 구분자
loader = CSVLoader(
    file_path='data.csv',
    csv_args={'delimiter': '|'}
)


## 💡 핵심 정리

### CSVLoader 기본 사용

```python
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path="data.csv")
docs = loader.load()
```

### 주요 옵션

1. **커스텀 필드명**
   ```python
   loader = CSVLoader(
       file_path="data.csv",
       csv_args={"fieldnames": ["컬럼1", "컬럼2", "컬럼3"]}
   )
   ```

2. **다른 구분자 사용**
   ```python
   loader = CSVLoader(
       file_path="data.tsv",
       csv_args={"delimiter": "\t"}
   )
   ```

3. **소스 컬럼 지정**
   ```python
   loader = CSVLoader(
       file_path="data.csv",
       source_column="id"  # id 컬럼을 source로 사용
   )
   ```

### 실전 팁

1. **대용량 CSV 처리**: `lazy_load()` 사용
   ```python
   for doc in loader.lazy_load():
       process(doc)  # 한 번에 하나씩 처리
   ```

2. **데이터 필터링**: 로드 후 Python으로 필터링
   ```python
   filtered = [doc for doc in docs if condition(doc)]
   ```

3. **인코딩 문제**: `encoding` 옵션 지정
   ```python
   loader = CSVLoader(
       file_path="data.csv",
       csv_args={"encoding": "utf-8"}
   )
   ```

### CSV vs DataFrame

| 작업 | CSVLoader | pandas DataFrame |
|------|-----------|------------------|
| **RAG용 문서화** | ✅ 적합 | ❌ 부적합 |
| **데이터 분석** | ❌ 부적합 | ✅ 적합 |
| **행 단위 처리** | ✅ 자동 | 수동 변환 필요 |
| **메타데이터 관리** | ✅ 자동 | 수동 추가 필요 |

**선택 가이드**: RAG 시스템이나 LLM 입력용이면 CSVLoader, 데이터 분석이나 변환 작업이면 pandas 사용


## 다음 단계

CSV 로더를 학습했으니, 다음은 Excel 파일 로더를 학습하겠습니다.

- **04-Excel-Loader**: Excel 파일 로딩
- **05-Word-Loader**: Word 문서 로딩
- **06-PowerPoint-Loader**: PowerPoint 로딩
